
# Hiking Route Optimizer — Extended QUBO / QAOA Formulation

Original problem: choose 2 POIs (one per time slot) from the Benasque area
that maximises scenic value while minimising travel distance.

Extensions added in this version (matching the tutorial's penalty style):
  1. **Altitude penalty**: iscourage routes that climb too steeply between stops
  2. **Gear-mismatch penalty**: each POI has a required gear level (Urban < Trail < Mountain < Snow).  If the user's gear is below what a POI requires, that POI is penalised
  3. **24-hour budget** (soft) the total travel + visit time must not exceed 24 h Implemented as a soft quadratic penalty no slack variables needed.

In [1]:
import itertools
from collections import defaultdict

import numpy as np
from qiskit.primitives import StatevectorSampler
from qiskit_algorithms.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer

**Uploading data and problem parameters**

In [2]:
locations = {
    1: ("Pico Aneto", "Peak", 3404, "Snow", "Snow"),
    2: ("Tuca Alba", "Peak", 3122, "Snow", "Mountain"),
    3: ("Benasque", "Town", 1135, "Urban", "Urban"),
    4: ("Cerler", "Town", 1530, "Urban", "Urban"),
    5: ("Ancils", "Town", 1140, "Urban", "Urban"),
    6: ("Portillón de Benás", "Landmark", 2444, "Mountain", "Trail"),
    7: ("Ski resort", "Snow", 1530, "Snow", "Trail"),
    8: ("Baños de Benasque", "Resting area", 1550, "Urban", "Urban"),
    9: ("Hospital de Benasque", "Resting area", 1747, "Urban", "Urban"),
    10: ("Forau d'aiguallut", "Landmark", 2020, "Mountain", "Trail"),
    11: ("Tres Cascadas", "Landmark", 1900, "Mountain", "Trail"),
    12: ("Salvaguardia", "Peak", 2736, "Snow", "Mountain"),
    13: ("Tuca Maladeta", "Peak", 3312, "Snow", "Mountain"),
    14: ("Cap Llauset", "Refugio", 2425, "Snow", "Mountain"),
    15: ("Ibón Cregüeña", "Lake", 2632, "Snow", "Mountain"),
    16: ("Batisielles", "Lake", 2216, "Mountain", "Trail"),
    17: ("Eriste", "Town", 1089, "Urban", "Urban"),
    18: ("Ibón Eriste", "Lake", 2407, "Snow", "Mountain"),
    19: ("Tempestades", "Peak", 3289, "Snow", "Snow"),
    20: ("La Besurta", "Resting area", 1860, "Trail", "Trail"),
    21: ("La Renclusa", "Refugio", 2160, "Snow", "Trail"),
    22: ("Escaleta", "Lake", 2630, "Snow", "Mountain"),
    23: ("Mulleres", "Peak", 3013, "Snow", "Snow"),
    24: ("Salterillo", "Lake", 2460, "Snow", "Mountain"),
    25: ("Tres Barrancos", "Landmark", 1460, "Trail", "Trail"),
}

In [3]:
# Base camp (node 0) — Benasque town (index 3 in the table, but we keep 0 here for easier indexing in the distance matrix)
BASE_ALTITUDE = 1135  # metres (Benasque)
BASE_CAMP = 0  # index in the distance matrix
GEAR_LEVEL = {"Urban": 0, "Trail": 1, "Mountain": 2, "Snow": 3}

In [4]:
# Toy model parameters
pois = [4, 5, 6]  # Cerler, Ancils, Portillón de Benás

# Scenic values for each POI (higher = more rewarding)
scenic = {
    4: 4.0,  # Cerler            — ski village, nice views
    5: 3.5,  # Ancils            — quiet hamlet
    6: 8.0,  # Portillón de Benás — mountain pass, panoramic views
}
SEASON = "summer"

# The hiker's gear level.
USER_GEAR = GEAR_LEVEL["Mountain"]

In [5]:
# Travel-time matrix in hours. Row/col order: [0=Benasque, 1=Cerler, 2=Ancils, 3=Portillón]
# Used for both the objective function (minimise travel time) and the budget penalty.
travel_time = np.array(
    [
        [0.00, 1.20, 0.45, 2.50],  # from Benasque
        [1.20, 0.00, 1.30, 3.00],  # from Cerler
        [0.45, 1.30, 0.00, 1.42],  # from Ancils
        [2.50, 3.00, 1.42, 0.00],  # from Portillón
    ],
    dtype=float,
)

# Visit time at each POI (hours) — needed for the time-budget penalty.
visit_time = {4: 1.0, 5: 0.5, 6: 2.0}  # Cerler, Ancils, Portillón

In [6]:
def gear_required(loc_id: int, season: str = "summer") -> int:
    """Return the numeric gear level required to visit a location."""
    col = 3 if season == "summer" else 4  # index into the tuple
    return GEAR_LEVEL[locations[loc_id][col]]


def altitude(loc_id: int) -> float:
    """Return the altitude (m) of a location."""
    return locations[loc_id][2]


def is_refugio(loc_id: int) -> bool:
    """Return True if the location is a Refugio (overnight stay possible)."""
    return locations[loc_id][1] == "Refugio"

**Number of time slots**

Set N_SLOTS = 2 for a standard day hike (one outbound POI + one return POI).Set N_SLOTS = 3 to model an overnight trip where the middle slot can be used for a refugio stay, letting the optimizer consider a three-stop sequence.

## Build QUBO

In [7]:
N_SLOTS = 2  # ← change to 3 to compare the overnight variant
slots = list(range(1, N_SLOTS + 1))

**Build penalty coefficients**

In [8]:
A = 12.0  # "one POI per slot" hard constraint
B = 10.0  # "no revisit" hard constraint
C = 6.0  # altitude-gain soft penalty
D = 8.0  # gear-mismatch soft penalty
E = 5.0  # time-budget soft penalty
F = 50.0  # Snow ↔ Mountain gear hard constraint (large → effectively forbidden)
# Must be larger than any achievable objective gain to truly forbid.

T_MAX = 24.0  # normal maximum hike duration (hours)
T_MAX_REFUGIO = 48.0  # extended budget when a Refugio is in the route

# Maximum altitude gain allowed between consecutive stops before penalising.
MAX_ALT_GAIN = 800  # metres

In [9]:
qp = QuadraticProgram("benasque_hiking_route")
for i in pois:
    for p in slots:
        qp.binary_var(name=f"x_{i}_{p}")

# We accumulate coefficients in three dicts and add them to the QP at the end,
# exactly as in your original script.
linear = defaultdict(float)
quadratic = defaultdict(float)
constant = 0.0

In [10]:
# Point Of Interest mapping

# dist / travel_time rows/cols are indexed as [0=base, 1=pois[0], 2=pois[1], …]
poi_to_dist_idx = {poi: k + 1 for k, poi in enumerate(pois)}


def d(a: int, b: int) -> float:
    """Travel time (hours) from node a to node b."""
    ai = 0 if a == BASE_CAMP else poi_to_dist_idx[a]
    bi = 0 if b == BASE_CAMP else poi_to_dist_idx[b]
    return travel_time[ai][bi]

The first objective is to minimise distance − scenic reward

In [11]:
for i in pois:
    # First slot: depart from base camp
    linear[f"x_{i}_1"] += d(BASE_CAMP, i) - scenic[i]
    # Last slot: return to base camp
    linear[f"x_{i}_{N_SLOTS}"] += d(i, BASE_CAMP) - scenic[i]
    # Middle slots (only exist when N_SLOTS >= 3): scenic reward only
    for p in slots[1:-1]:
        linear[f"x_{i}_{p}"] += -scenic[i]

# Travel cost between every pair of consecutive slots
for p, q in zip(slots, slots[1:]):  # (1,2), (2,3), …
    for i in pois:
        for j in pois:
            quadratic[(f"x_{i}_{p}", f"x_{j}_{q}")] += d(i, j)

**Implementing constraints**

1. I want to go to exactly one point of intereset per time slot (can't be in multiple place at same time). The implemented penalty term is: 
$$A \left[ \sum_i x_{i,p} \;+\; 2 \sum_{i<j} x_{i,p} x_{j,p} \;-\; 1 \right] \;+\; A$$

2. I do not want to revisit places where I have already been, in this case the penalty term is given by: $$B \sum_{p<q} x_{i,p} \, x_{i,q}$$ for every POI i

3.  Penalise large altitude gain in one step. We penalise any pair (slot-1 POI i, slot-2 POI j) whose altitude ifference exceeds MAX_ALT_GAIN. In this case the penalty is implemented as a soft quadratic penalty. We implemented ascent penalty between slots as:
$$\sum_{i,j} C \left( \big( (\mathrm{alt}(j) - \mathrm{alt}(i))_+ - \mathrm{MAX\_ALT\_GAIN} \big)_+ \right)^2 \; x_{i,1} x_{j,2}$$, and ascent penalties from base as $$\sum_i C \left( \big( (\mathrm{alt}(i) - \mathrm{BASE\_ALTITUDE})_+ - \mathrm{MAX\_ALT\_GAIN} \big)_+ \right)^2 \; x_{i,1}$$


4. Gear mismatch: visiting a "Snow" POI with only "Urban" gear is dangerous. We penalise each variable whose POI requires higher gear than the user has. In this case we implemented a linear soft penalty on each variable: $$\sum_{i,p} D \left( \mathrm{required\_gear\_level}_i - \mathrm{user\_gear\_level} \right)_+ \; x_{i,p}$$

5. Time penalty: For each ordered sequence of POIs across all N_SLOTS slots, the total hike time is compared against a budget that depends on whether any POI in that sequence is a Refugio: budget = T_MAX_REFUGIO (48 h)  if any POI in the sequence is a Refugio budget = T_MAX (24 h)  otherwise.

The excess beyond the applicable budget is penalised with a soft quadratic term on the product of all variables in the sequence 

In [14]:
for p in slots:
    constant += A  # +A from expanding (…−1)²
    for i in pois:
        linear[f"x_{i}_{p}"] += -A  # −2A·1·x from cross term (2·A·(-1)·x)
        # Wait — full expansion:
        # (Σx − 1)² = Σx² + 2Σ_{i<j}x_ix_j − 2Σx + 1
        # For binary: x² = x, so diagonal += (1−2)·A = −A
    for i, j in itertools.combinations(pois, 2):
        quadratic[(f"x_{i}_{p}", f"x_{j}_{p}")] += 2 * A  # off-diagonal cross terms

for i in pois:
    for p, q in itertools.combinations(slots, 2):
        quadratic[(f"x_{i}_{p}", f"x_{i}_{q}")] += B

for i in pois:
    # Base → first POI altitude gain
    gain_base_i = max(0.0, altitude(i) - BASE_ALTITUDE)
    excess_base_i = max(0.0, gain_base_i - MAX_ALT_GAIN)
    if excess_base_i > 0:
        linear[f"x_{i}_1"] += C * excess_base_i**2

    for j in pois:
        # Altitude gain across every pair of consecutive slots (p → p+1)
        gain_ij = max(0.0, altitude(j) - altitude(i))
        excess_ij = max(0.0, gain_ij - MAX_ALT_GAIN)
        if excess_ij > 0:
            for p, q in zip(slots, slots[1:]):
                quadratic[(f"x_{i}_{p}", f"x_{j}_{q}")] += C * excess_ij**2

for i in pois:
    required = gear_required(i, season=SEASON)
    deficit = max(0, required - USER_GEAR)
    if deficit > 0:
        for p in slots:
            # Linear penalty: fires independently for each slot
            linear[f"x_{i}_{p}"] += D * deficit

for i in pois:
    for j in pois:
        # Determine the effective time budget for this (i, j) pair:
        # extend to 48 h if either POI is a Refugio.
        budget = T_MAX_REFUGIO if (is_refugio(i) or is_refugio(j)) else T_MAX

        # Evaluate every consecutive-slot pair that involves i then j.
        for p, q in zip(slots, slots[1:]):
            # Total time for *this leg* only (base↔POI only on first/last legs)
            if p == slots[0]:
                leg_time = d(BASE_CAMP, i) + visit_time[i] + d(i, j)
            else:
                leg_time = visit_time[i] + d(i, j)
            if q == slots[-1]:
                leg_time += visit_time[j] + d(j, BASE_CAMP)

            excess = max(0.0, leg_time - budget)
            if excess > 0:
                quadratic[(f"x_{i}_{p}", f"x_{j}_{q}")] += E * excess**2

In [15]:
# convert to Ising
qp.minimize(constant=constant, linear=dict(linear), quadratic=dict(quadratic))
op, offset = qp.to_ising()

## Convert from QUBO to QAOA

In [19]:
def decode_solution(xvec):
    """Map the binary solution vector back to an ordered list of POIs.

    Returns [BASE_CAMP, poi_slot1, poi_slot2, …, BASE_CAMP].
    A slot with no POI selected is represented as None.
    """
    var_names = [f"x_{i}_{p}" for i in pois for p in slots]
    sol = {name: int(round(val)) for name, val in zip(var_names, xvec)}
    route_pois = []
    for p in slots:
        chosen = next((i for i in pois if sol.get(f"x_{i}_{p}") == 1), None)
        route_pois.append(chosen)
    return [BASE_CAMP] + route_pois + [BASE_CAMP]


def route_info(route):
    """Return a human-readable summary of a decoded route."""
    lines = []
    for idx, node in enumerate(route):
        if node == BASE_CAMP:
            lines.append(f"  Base camp (Benasque, {BASE_ALTITUDE} m)")
        else:
            name = locations[node][0]
            alt = altitude(node)
            req = locations[node][3 if SEASON == "summer" else 4]
            refugio = " [REFUGIO — overnight possible]" if is_refugio(node) else ""
            lines.append(
                f"  Slot {idx}: POI {node}: {name} ({alt} m)"
                f" — needs {req} gear{refugio}"
            )
    return "\n".join(lines)


def total_hike_time(route):
    """Compute the total hike duration (hours) for a decoded route."""
    stops = route[1:-1]  # strip the two base-camp bookends
    if None in stops:
        return float("nan")
    total = 0.0
    prev = BASE_CAMP
    for s in stops:
        total += d(prev, s) + visit_time[s]
        prev = s
    total += d(prev, BASE_CAMP)
    return total


def effective_budget(route):
    """Return the applicable time budget (24 h or 48 h) for a route."""
    stops = route[1:-1]
    if any(s is not None and is_refugio(s) for s in stops):
        return T_MAX_REFUGIO
    return T_MAX

In [20]:
sampler = StatevectorSampler()

# Exact solution using the NumPy eigensolver (classical brute-force)
exact_result = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)

# QAOA (reps=2, COBYLA optimiser)
qaoa_result = MinimumEigenOptimizer(
    QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=2)
).solve(qp)

In [21]:
def print_result(label, result):
    route = decode_solution(result.x)
    hike_time = total_hike_time(route)
    budget = effective_budget(route)
    print(f"\n{'─'*55}")
    print(f"  Solver : {label}")
    print(f"  Cost   : {result.fval:.2f}")
    print(f"  Route  :\n{route_info(route)}")
    if not np.isnan(hike_time):
        budget_ok = (
            f"✓ within {budget:.0f} h budget"
            if hike_time <= budget
            else f"✗ OVER {budget:.0f} h BUDGET"
        )
        print(f"  Time   : {hike_time:.1f} h  ({budget_ok})")
    else:
        print("  Time   : N/A (incomplete route)")


print_result("EXACT (NumPy)", exact_result)
print_result("QAOA  (reps=2)", qaoa_result)

print(f"\n{'─'*55}")
print("Configuration:")
print(f"  N_SLOTS               = {N_SLOTS}")
print(f"  Season                = {SEASON}")
print(
    f"  User gear             = {USER_GEAR} ({[k for k,v in GEAR_LEVEL.items() if v==USER_GEAR][0]})"
)
print("Penalty weights used:")
print(f"  A (one POI per slot)  = {A}")
print(f"  B (no revisit)        = {B}")
print(f"  C (altitude gain)     = {C}  [threshold: {MAX_ALT_GAIN} m]")
print(f"  D (gear mismatch)     = {D}")
print(f"  E (time budget)       = {E}  [normal: {T_MAX} h, refugio: {T_MAX_REFUGIO} h]")
print(f"  F (Snow↔Mountain ban) = {F}  [hard constraint]")


───────────────────────────────────────────────────────
  Solver : EXACT (NumPy)
  Cost   : -4.55
  Route  :
  Base camp (Benasque, 1135 m)
  Slot 1: POI 5: Ancils (1140 m) — needs Urban gear
  Slot 2: POI 4: Cerler (1530 m) — needs Urban gear
  Base camp (Benasque, 1135 m)
  Time   : 4.5 h  (✓ within 24 h budget)

───────────────────────────────────────────────────────
  Solver : QAOA  (reps=2)
  Cost   : -4.55
  Route  :
  Base camp (Benasque, 1135 m)
  Slot 1: POI 4: Cerler (1530 m) — needs Urban gear
  Slot 2: POI 5: Ancils (1140 m) — needs Urban gear
  Base camp (Benasque, 1135 m)
  Time   : 4.5 h  (✓ within 24 h budget)

───────────────────────────────────────────────────────
Configuration:
  N_SLOTS               = 2
  Season                = summer
  User gear             = 2 (Mountain)
Penalty weights used:
  A (one POI per slot)  = 12.0
  B (no revisit)        = 10.0
  C (altitude gain)     = 6.0  [threshold: 800 m]
  D (gear mismatch)     = 8.0
  E (time budget)       = 5.0